# Courbes d'entraînement — Chess Post-Training

Ce notebook trace l'évolution de la **loss** et des **métriques d'évaluation** à partir des journaux CSV dans `post_training/logs/`.

Métriques alignées avec le protocole du README / `evaluate.py` :
- taux de coups légaux (top-1 / top-2 / top-3)
- taux de fallback aléatoire
- accuracy vs Stockfish (top-1 / top-3)
- ACPL et score composite

Phases couvertes : **SFT** puis **RL GRPO**.

In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Le notebook est dans post_training/logs/ (à côté des CSV).
# On couvre aussi un cwd à la racine du repo ou ailleurs.
candidates = [
    Path("."),
    Path(__vsc_ipynb_file__).resolve().parent if "__vsc_ipynb_file__" in globals() else None,
    Path("logs"),
    Path("post_training/logs"),
    Path("../logs"),
    Path("../post_training/logs"),
]
LOGS = next(
    (p for p in candidates if p is not None and (p / "sft_steps.csv").exists()),
    None,
)
if LOGS is None:
    raise FileNotFoundError(
        "sft_steps.csv introuvable. cwd="
        + str(Path.cwd().resolve())
        + " — ouvre le notebook depuis post_training/logs/"
    )
print("Logs:", LOGS.resolve())

sft_steps = pd.read_csv(LOGS / "sft_steps.csv")
sft_val = pd.read_csv(LOGS / "sft_val.csv")
rl_steps = pd.read_csv(LOGS / "rl_grpo_steps.csv")
rl_val = pd.read_csv(LOGS / "rl_grpo_val.csv")

print("SFT steps:", sft_steps.shape, "| SFT val:", sft_val.shape)
print("RL steps:", rl_steps.shape, "| RL val:", rl_val.shape)
sft_val.head()

StopIteration: 

## 1. SFT — évolution de la loss (train)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)

# Total loss + CE
ax = axes[0, 0]
ax.plot(sft_steps["step"], sft_steps["loss"], label="loss", alpha=0.85)
ax.plot(sft_steps["step"], sft_steps["ce_best"], label="ce_best", alpha=0.85)
ax.set_ylabel("loss")
ax.set_title("SFT train — loss totale & CE")
ax.legend()
ax.grid(True, alpha=0.3)

# Loss components
ax = axes[0, 1]
ax.plot(sft_steps["step"], sft_steps["kl_uniform_legal"], label="kl_uniform_legal")
ax.plot(sft_steps["step"], sft_steps["illegal_penalty"], label="illegal_penalty")
ax.plot(sft_steps["step"], sft_steps["kl_ref"], label="kl_ref")
ax.set_ylabel("term")
ax.set_title("SFT train — composantes de la loss")
ax.legend()
ax.grid(True, alpha=0.3)

# Legal prob
ax = axes[1, 0]
ax.plot(sft_steps["step"], sft_steps["mean_legal_prob"], color="C2")
ax.set_xlabel("step")
ax.set_ylabel("mean_legal_prob")
ax.set_title("SFT train — masse de probabilité légale")
ax.grid(True, alpha=0.3)

# LR + grad norm
ax = axes[1, 1]
ax.plot(sft_steps["step"], sft_steps["lr"], color="C3", label="lr")
ax.set_xlabel("step")
ax.set_ylabel("learning rate", color="C3")
ax.tick_params(axis="y", labelcolor="C3")
ax2 = ax.twinx()
ax2.plot(sft_steps["step"], sft_steps["grad_norm"], color="C4", alpha=0.7, label="grad_norm")
ax2.set_ylabel("grad_norm", color="C4")
ax2.tick_params(axis="y", labelcolor="C4")
ax.set_title("SFT train — LR & grad norm")
ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## 2. SFT — métriques d'évaluation (validation)

Seuils du *SFT gate* (README / `eval_local.py`) : top-1 légal ≥ 92 %, top-3 légal ≥ 98 %.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
x = sft_val["step"]

# Val loss
ax = axes[0, 0]
ax.plot(x, sft_val["val_loss"], marker="o", label="val_loss")
ax.plot(x, sft_val["val_ce_loss"], marker="o", label="val_ce_loss")
ax.set_ylabel("loss")
ax.set_title("SFT val — loss")
ax.legend()
ax.grid(True, alpha=0.3)

# Legal rates (README eval metrics)
ax = axes[0, 1]
ax.plot(x, sft_val["top1_legal_rate"], marker="o", label="top1_legal")
ax.plot(x, sft_val["top2_legal_rate"], marker="o", label="top2_legal")
ax.plot(x, sft_val["top3_legal_rate"], marker="o", label="top3_legal")
ax.axhline(0.92, color="C0", ls="--", alpha=0.5, label="gate top1 ≥ 0.92")
ax.axhline(0.98, color="C2", ls="--", alpha=0.5, label="gate top3 ≥ 0.98")
ax.set_ylabel("rate")
ax.set_ylim(0.9, 1.005)
ax.set_title("SFT val — légalité (top-k)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# SF accuracy + fallback
ax = axes[1, 0]
ax.plot(x, sft_val["top1_sf_accuracy"], marker="o", label="top1_sf_acc")
ax.plot(x, sft_val["top3_sf_accuracy"], marker="o", label="top3_sf_acc")
ax.plot(x, sft_val["fallback_random_rate"], marker="o", label="fallback_random")
ax.set_xlabel("step")
ax.set_ylabel("rate")
ax.set_title("SFT val — SF accuracy & fallback aléatoire")
ax.legend()
ax.grid(True, alpha=0.3)

# ACPL + composite
ax = axes[1, 1]
ax.plot(x, sft_val["acpl"], marker="o", color="C3", label="acpl")
ax.set_xlabel("step")
ax.set_ylabel("ACPL", color="C3")
ax.tick_params(axis="y", labelcolor="C3")
ax2 = ax.twinx()
ax2.plot(x, sft_val["composite_score"], marker="s", color="C5", label="composite")
ax2.set_ylabel("composite_score", color="C5")
ax2.tick_params(axis="y", labelcolor="C5")
ax.set_title("SFT val — ACPL & composite score")
ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

print("Dernière eval SFT:")
print(sft_val.iloc[-1][[
    "step", "val_loss", "top1_legal_rate", "top3_legal_rate",
    "fallback_random_rate", "top1_sf_accuracy", "acpl", "composite_score", "sft_gate_passed"
]].to_string())

## 3. RL GRPO — évolution de la loss (train)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)

ax = axes[0, 0]
ax.plot(rl_steps["step"], rl_steps["loss"], marker="o", label="loss")
ax.plot(rl_steps["step"], rl_steps["grpo_loss"], marker="o", label="grpo_loss")
ax.plot(rl_steps["step"], rl_steps["q_distill_loss"], marker="o", label="q_distill_loss")
ax.set_ylabel("loss")
ax.set_title("RL train — losses")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.plot(rl_steps["step"], rl_steps["kl_ref"], marker="o", label="kl_ref")
ax.plot(rl_steps["step"], rl_steps["illegal_pen"], marker="o", label="illegal_pen")
ax.set_ylabel("term")
ax.set_title("RL train — régularisation")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
ax.plot(rl_steps["step"], rl_steps["mean_group_reward"], marker="o", label="mean_group_reward")
ax.plot(rl_steps["step"], rl_steps["top1_cand_reward"], marker="o", label="top1_cand_reward")
ax.plot(rl_steps["step"], rl_steps["group_legal_ratio"], marker="o", label="group_legal_ratio")
ax.set_xlabel("step")
ax.set_ylabel("value")
ax.set_title("RL train — rewards & légalité groupe")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax.plot(rl_steps["step"], rl_steps["lr"], color="C3", marker="o", label="lr")
ax.set_xlabel("step")
ax.set_ylabel("learning rate", color="C3")
ax.tick_params(axis="y", labelcolor="C3")
ax2 = ax.twinx()
ax2.plot(rl_steps["step"], rl_steps["grad_norm"], color="C4", marker="s", alpha=0.7, label="grad_norm")
ax2.set_ylabel("grad_norm", color="C4")
ax2.tick_params(axis="y", labelcolor="C4")
ax.set_title("RL train — LR & grad norm")
ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## 4. RL GRPO — métriques d'évaluation (validation)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
x = rl_val["step"]

ax = axes[0, 0]
ax.plot(x, rl_val["val_loss"], marker="o", label="val_loss")
ax.plot(x, rl_val["val_ce_loss"], marker="o", label="val_ce_loss")
ax.set_ylabel("loss")
ax.set_title("RL val — loss")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.plot(x, rl_val["top1_legal_rate"], marker="o", label="top1_legal")
ax.plot(x, rl_val["top3_legal_rate"], marker="o", label="top3_legal")
ax.axhline(0.92, color="C0", ls="--", alpha=0.5)
ax.axhline(0.98, color="C1", ls="--", alpha=0.5)
ax.set_ylabel("rate")
ax.set_ylim(0.9, 1.005)
ax.set_title("RL val — légalité")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
ax.plot(x, rl_val["top1_sf_accuracy"], marker="o", label="top1_sf_acc")
ax.plot(x, rl_val["top3_sf_accuracy"], marker="o", label="top3_sf_acc")
ax.plot(x, rl_val["fallback_random_rate"], marker="o", label="fallback_random")
ax.set_xlabel("step")
ax.set_ylabel("rate")
ax.set_title("RL val — SF accuracy & fallback")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax.plot(x, rl_val["acpl"], marker="o", color="C3", label="acpl")
ax.set_xlabel("step")
ax.set_ylabel("ACPL", color="C3")
ax.tick_params(axis="y", labelcolor="C3")
ax2 = ax.twinx()
ax2.plot(x, rl_val["composite_score"], marker="s", color="C5", label="composite")
ax2.set_ylabel("composite_score", color="C5")
ax2.tick_params(axis="y", labelcolor="C5")
ax.set_title("RL val — ACPL & composite")
ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

print("Dernière eval RL:")
print(rl_val.iloc[-1][[
    "step", "val_loss", "top1_legal_rate", "top3_legal_rate",
    "fallback_random_rate", "top1_sf_accuracy", "acpl", "composite_score", "sft_gate_passed"
]].to_string())

## 5. Synthèse base → SFT → RL

Comparaison des métriques clés du README (légalité, fallback, qualité vs Stockfish).

In [ ]:
metrics = [
    "val_loss", "top1_legal_rate", "top3_legal_rate",
    "fallback_random_rate", "top1_sf_accuracy", "top3_sf_accuracy",
    "acpl", "composite_score",
]

summary = pd.DataFrame({
    "base": sft_val.iloc[0][metrics].values,
    "sft_best": sft_val.iloc[-1][metrics].values,
    "rl_final": rl_val.iloc[-1][metrics].values,
}, index=metrics)

display(summary.round(4))

# Bar chart before/after for the main eval rates
rates = ["top1_legal_rate", "top3_legal_rate", "top1_sf_accuracy", "fallback_random_rate"]
fig, ax = plt.subplots(figsize=(10, 4))
x_pos = range(len(rates))
width = 0.25
ax.bar([i - width for i in x_pos], summary.loc[rates, "base"], width, label="base")
ax.bar(list(x_pos), summary.loc[rates, "sft_best"], width, label="après SFT")
ax.bar([i + width for i in x_pos], summary.loc[rates, "rl_final"], width, label="après RL")
ax.set_xticks(list(x_pos))
ax.set_xticklabels(rates, rotation=15)
ax.set_ylabel("rate")
ax.set_title("Évolution des métriques d'évaluation (base → SFT → RL)")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout()
plt.show()